# Обучение ODIN на NBV Stage2 Dataset (Multi-object, 3 Textures)
Этот ноутбук обучает ODIN на датасете NBV Stage 2 с примитивами (8 геометрических форм × 3 текстуры = 24 класса).

## Информация о датасете

**NBV Stage 2 Dataset**: https://www.kaggle.com/datasets/sergeykurchev/nbv-stage2-dataset

### Характеристики (проверено локально):
- **Objects**: 2-10 примитивов (8 классов: cube, sphere, cylinder, cone, torus, capsule, ellipsoid, pyramid)
- **Textures**: 3 типа (red, mixed, green)
- **Total Classes**: 24 (8 shapes × 3 textures)
- **Images**: 224×224 RGB + depth + masks
- **Cameras**: **5 кадров** на сцену (не ~20!)
- **cameras.json**: dict {"00000": {position, rotation, intrinsics}, ...} — ключ = индекс кадра
- **color_map.json**: list [{color, instance_id, category_id(1-8=primitive, 9=robot), texture_type}]
  - `category_name` всегда = `"target_object"` — **не информативно**!
  - Реальный класс = `NBV_PRIMITIVE_NAMES[category_id] + '_' + texture_type`
- **Purpose**: Multi-object active classification с разными текстурами

In [ ]:
import os
import subprocess
import sys
import urllib.request

# Базовая конфигурация проекта
CONFIG = {
    "ODIN_DIR": "my_odin",
    "ODIN_REPO_URL": "https://github.com/SergKurchev/my_odin.git",
    # Публичные веса из katefgroup/odin
    "ODIN_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/scannet_resnet_47.8_73.3_32k_1.5k.pth",
    "ODIN_WEIGHTS_PATH": "my_odin/models/odin_scannet_context.pth",
    "M2F_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/m2f_coco.pkl",
    "M2F_WEIGHTS_PATH": "my_odin/models/model_final_5c90d4.pkl",
}


## 1. Создание виртуального окружения и установка зависимостей

In [ ]:
# 1. Создаем venv (даже если ensurepip нет, структура папок создастся)
if not os.path.exists("venv"):
    print("Устанавливаю Python 3.10...")
    subprocess.run(["apt-get", "update", "-y"], check=False)
    subprocess.run(["apt-get", "install", "-y", "python3.10", "python3.10-venv", "python3.10-dev", "python3.10-distutils"], check=False)

    print("Создаю виртуальное окружение на Python 3.10...")
    # Флаг --without-pip предотвращает попытки использовать отсутствующий ensurepip
    subprocess.run(["python3.10", "-m", "venv", "venv", "--without-pip"], check=True)

    VENV_PYTHON_TMP = os.path.abspath("venv/bin/python")

    # Прямая установка pip через официальный скрипт
    print("Загружаю get-pip.py для ручной установки...")
    urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", "get-pip.py")

    print("Устанавливаю pip в виртуальное окружение...")
    subprocess.run([VENV_PYTHON_TMP, "get-pip.py"], check=True)

    # Чистим за собой
    if os.path.exists("get-pip.py"):
        os.remove("get-pip.py")
    print("Pip успешно установлен!")

# Теперь пути будут корректными
VENV_PYTHON = os.path.abspath("venv/bin/python")
VENV_PIP = os.path.abspath("venv/bin/pip")


def make_venv_env(extra=None):
    """
    Создаёт словарь переменных окружения с правильными путями к venv.

    КЛЮЧЕВОЕ ОТЛИЧИЕ от системной среды (reference):
    В системной среде pip/python берутся из PATH и уже знают о системных пакетах.
    В venv — нужно явно прописать VIRTUAL_ENV и PATH, иначе subprocess-ы,
    запущенные из подпроцессов (например, build isolation в pip),
    не будут знать, что они внутри venv и не найдут torch.
    """
    env = os.environ.copy()
    venv_dir = os.path.abspath("venv")
    env["VIRTUAL_ENV"] = venv_dir
    env["PATH"] = os.path.join(venv_dir, "bin") + ":" + env.get("PATH", "")
    # Убираем PYTHONPATH системы — он может мешать изоляции
    env.pop("PYTHONPATH", None)
    if extra:
        env.update(extra)
    return env


def clean_build_artifacts(directory):
    """
    Очищает старые артефакты сборки перед компиляцией.

    Ref-функция это делает явно — без очистки повторный запуск
    может завершиться ошибкой из-за конфликта старых .egg файлов.
    """
    import shutil
    import glob
    for d in ["build", "dist"]:
        path = os.path.join(directory, d)
        if os.path.exists(path):
            print(f"   Cleaning {path}...")
            shutil.rmtree(path)
    for egg in glob.glob(os.path.join(directory, "*.egg-info")):
        print(f"   Cleaning {egg}...")
        shutil.rmtree(egg)


def run_in_venv(cmd, cwd=None, env=None, check=True):
    """Обертка для запуска команд внутри созданного виртуального окружения."""
    if cmd[0] == "pip":
        cmd[0] = VENV_PIP
    elif cmd[0] == "python":
        cmd[0] = VENV_PYTHON

    print(f"Выполняю: {' '.join(cmd)}")
    subprocess.run(cmd, cwd=cwd, env=env, check=check)


def install_odin_dependencies():
    """Установка зависимостей ODIN в виртуальное окружение."""
    print("\n" + "=" * 80)
    print("Installing ODIN Dependencies into VENV")
    print("=" * 80)

    venv_env = make_venv_env()

    # 0. Клонирование репозитория
    if not os.path.exists(CONFIG["ODIN_DIR"]):
        print("\n0. Cloning ODIN...")
        subprocess.run(["git", "clone", "-q", CONFIG["ODIN_REPO_URL"], CONFIG["ODIN_DIR"]], check=True)

    # 1. PyTorch 2.2.0 + CUDA 12.1
    print("\n1. Installing PyTorch 2.2.0...")
    run_in_venv([
        "pip", "install", "-q", "torch==2.2.0", "torchvision==0.17.0",
        "--index-url", "https://download.pytorch.org/whl/cu121"
    ], env=venv_env)

    run_in_venv([
        "pip", "install", "-q", "torch-scatter",
        "-f", "https://data.pyg.org/whl/torch-2.2.0+cu121.html"
    ], env=venv_env)

    # 2. NumPy < 2 и Pillow
    print("\n2. Installing NumPy < 2 + Pillow...")
    run_in_venv(["pip", "install", "-q", "numpy<2", "--force-reinstall"], env=venv_env)
    run_in_venv(["pip", "install", "-q", "Pillow>=10.2.0"], env=venv_env)

    # 3. Clean requirements (через Python — надежнее чем sed)
    print("\n3. Cleaning ODIN requirements...")
    req_path = os.path.join(CONFIG["ODIN_DIR"], "requirements.txt")

    with open(req_path, 'r') as f:
        lines = f.readlines()

    with open(req_path, 'w') as f:
        for line in lines:
            line_clean = line.strip().lower()
            # Пропускаем все проблемные пакеты
            if any(x in line_clean for x in ["waspinator", "detectron2", "pytorch3d"]):
                print(f"   Skipping: {line.strip()}")
                continue
            # Фикс для старого PyYAML
            if "pyyaml==5.3.1" in line_clean:
                f.write("pyyaml>=5.4.1\n")
            else:
                f.write(line)

    # 4. Build tools + Modern COCO API (нужны перед requirements.txt)
    print("\n4. Installing Build Tools & Modern dependencies...")
    run_in_venv(["pip", "install", "-q", "cython", "setuptools", "wheel", "pycocotools"], env=venv_env)

    # 5. ODIN requirements + ninja, fvcore, iopath
    print("\n5. Installing ODIN requirements...")
    run_in_venv(["pip", "install", "-q", "-r", req_path], env=venv_env)
    run_in_venv(["pip", "install", "-q", "ninja", "fvcore", "iopath"], env=venv_env)

    # 6. Detectron2
    # КЛЮЧЕВОЕ ОТЛИЧИЕ от системной среды:
    # В системной среде pip видит torch в системных пакетах Kaggle.
    # В нашем venv pip должен использовать torch из venv,
    # поэтому нужен --no-build-isolation (запрет временного пустого build env)
    # + явно прокидываем venv_env с правильным VIRTUAL_ENV и PATH.
    print("\n6. Installing Detectron2...")
    run_in_venv([
        "pip", "install", "-q", "--no-build-isolation",
        "git+https://github.com/facebookresearch/detectron2.git"
    ], env=venv_env)

    # 7. PyTorch3D
    print("\n7. Installing PyTorch3D (with CUDA)...")
    cuda_env = make_venv_env({
        "FORCE_CUDA": "1",
        "TORCH_CUDA_ARCH_LIST": "6.0;7.0;7.5;8.0;8.6",
    })
    run_in_venv([
        "pip", "install", "-q", "--no-build-isolation",
        "git+https://github.com/facebookresearch/pytorch3d.git"
    ], env=cuda_env)

    # 8. Критически важно: переустановка правильных версий NumPy и OpenCV
    # (другие пакеты могут их обновить в процессе установки)
    print("\n8. Re-installing correct NumPy + OpenCV...")
    run_in_venv(["pip", "uninstall", "-y", "-q", "numpy"], env=venv_env)
    run_in_venv(["pip", "install", "-q", "numpy==1.26.4"], env=venv_env)
    run_in_venv(["pip", "install", "-q", "opencv-python-headless==4.8.0.76"], env=venv_env)

    # 9. Компиляция CUDA-ядер pointops2
    # КЛЮЧЕВОЕ ОТЛИЧИЕ:
    # Ref-функция делает os.chdir() + setup.py install --user.
    # --user ставит в ~/.local, откуда системный python видит, но venv — нет!
    # Мы передаём cwd= и VENV_PYTHON напрямую, БЕЗ --user.
    # Очистка артефактов — как в ref-функции.
    print("\n9. Compiling CUDA kernels (pointops2)...")
    pointops_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "libs", "pointops2"))
    clean_build_artifacts(pointops_dir)
    run_in_venv(["python", "setup.py", "install"], cwd=pointops_dir, env=cuda_env)

    # 10. Компиляция CUDA-ядер deformable attention
    print("\n10. Compiling CUDA kernels (deformable attention)...")
    deform_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "odin", "modeling", "pixel_decoder", "ops"))
    clean_build_artifacts(deform_dir)
    run_in_venv(["python", "setup.py", "build", "install"], cwd=deform_dir, env=cuda_env)

    print("\n" + "=" * 80)
    print("Installation complete!")
    print("=" * 80)


def download_weights():
    """Скачивание предобученных весов с публичных URL (katefgroup/odin)."""
    print("\n" + "=" * 80)
    print("Downloading Weights")
    print("=" * 80)

    os.makedirs(os.path.dirname(CONFIG["ODIN_WEIGHTS_PATH"]), exist_ok=True)

    def download_file(url, dest_path, min_size_mb=50):
        """Скачивает файл и проверяет целостность по размеру."""
        size_mb = os.path.getsize(dest_path) / (1024 * 1024) if os.path.exists(dest_path) else 0
        if size_mb >= min_size_mb:
            print(f"   ✓ Файл уже скачан ({size_mb:.0f} MB): {dest_path}")
            return
        if os.path.exists(dest_path):
            print(f"   ⚠️  Файл повреждён ({size_mb:.1f} MB), удаляю...")
            os.remove(dest_path)
        print(f"   Скачиваю {url} ...")
        ret = os.system(f"wget --tries=3 --retry-connrefused -c '{url}' -O '{dest_path}'")
        if ret != 0:
            print(f"   ❌ wget завершился с кодом {ret}, пробую curl...")
            os.system(f"curl -L --retry 3 '{url}' -o '{dest_path}'")
        size_mb = os.path.getsize(dest_path) / (1024 * 1024) if os.path.exists(dest_path) else 0
        print(f"   ✓ Скачано: {dest_path} ({size_mb:.0f} MB)")
        if size_mb < min_size_mb:
            raise RuntimeError(f"Файл слишком мал ({size_mb:.1f} MB) — скачивание провалилось!")

    print("\n1. Downloading ODIN weights (ScanNet ResNet50)...")
    download_file(CONFIG["ODIN_WEIGHTS_URL"], CONFIG["ODIN_WEIGHTS_PATH"], min_size_mb=100)

    print("\n2. Downloading Mask2Former weights (M2F COCO ResNet)...")
    download_file(CONFIG["M2F_WEIGHTS_URL"], CONFIG["M2F_WEIGHTS_PATH"], min_size_mb=50)

    print("\nWeights ready!")


# Запускаем сборку
install_odin_dependencies()
download_weights()

## 2. Диагностика датасета и создание splits

NBV Stage2 датасет имеет следующую структуру (проверено локально):
- **24 класса**: 8 примитивов × 3 текстуры (red, mixed, green)
- **5 кадров** на сэмпл (00000..00004)
- **cameras.json** — dict с ключами-индексами кадров, НЕТ поля `frame_index` внутри
- **color_map.json** — list, класс = `category_id + texture_type` (не `category_name`!)
- **category_id**: 1=cube, 2=sphere, 3=cylinder, 4=cone, 5=torus, 6=capsule, 7=ellipsoid, 8=pyramid, 9=robot (фон)

In [ ]:
# Диагностика NBV Stage2 датасета и создание splits
import json
from pathlib import Path

# ВАЖНО: Kaggle монтирует датасеты в /kaggle/input/datasets/<username>/<dataset-name>
DATASET_DIR = "/kaggle/input/datasets/sergeykurchev/nbv-stage2-dataset"
SPLITS_FILE = "./nbv_stage2_splits.json"  # Создадим локально

print("=== Диагностика NBV Stage2 датасета ===")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"DATASET_DIR exists: {os.path.exists(DATASET_DIR)}")

if os.path.exists(DATASET_DIR):
    # Собираем все sample_XXXXX директории
    entries = sorted([e for e in os.listdir(DATASET_DIR) if e.startswith("sample_")])
    print(f"\nНайдено {len(entries)} сэмплов")
    print(f"Первые 10: {entries[:10]}")
    print(f"Последние 10: {entries[-10:]}")

    # Создаем splits: 80% train, 10% val, 10% test
    import random
    random.seed(42)  # Фиксированный seed для воспроизводимости

    n_samples = len(entries)
    n_train = int(0.8 * n_samples)
    n_val = int(0.1 * n_samples)

    shuffled_entries = entries.copy()
    random.shuffle(shuffled_entries)

    splits = {
        "train": shuffled_entries[:n_train],
        "val": shuffled_entries[n_train:n_train + n_val],
        "test": shuffled_entries[n_train + n_val:]
    }

    with open(SPLITS_FILE, 'w') as f:
        json.dump(splits, f, indent=2)

    print(f"\n=== Созданы splits (random_state=42) ===")
    for split_name, ids in splits.items():
        print(f"{split_name}: {len(ids)} сэмплов")
        print(f"  Первые 5: {ids[:5]}")

    # Проверим структуру одного сэмпла
    sample_dir = Path(DATASET_DIR) / entries[0]
    if sample_dir.exists():
        print(f"\n=== Структура {entries[0]} ===")
        for item in sorted(sample_dir.iterdir()):
            if item.is_dir():
                count = len(list(item.iterdir()))
                print(f"  {item.name}/ ({count} файлов)")
            else:
                print(f"  {item.name}")

        # ================================================================
        # РЕАЛЬНЫЙ ФОРМАТ ДАТАСЕТА (проверено локально):
        # ----------------------------------------------------------------
        # cameras.json: DICT {"00000": {position, rotation(xyzw quat), intrinsics, ...}, ...}
        #   - НЕТ поля frame_index внутри — ключ словаря и есть индекс кадра
        #   - rotation: [x, y, z, w] — кватернион (4 элемента)
        #   - intrinsics: {fx, fy, cx, cy, width, height, near, far, fov_deg}
        #   - cx=112, cy=112 → img_w = cx*2 = 224 px ✓
        # ----------------------------------------------------------------
        # color_map.json: СПИСОК объектов, каждый:
        #   {color: [R,G,B], instance_id: N, category_id: 1-9,
        #    category_name: "target_object",  ← ВСЕГДА одинаково, НЕ информативно!
        #    texture_type: "red"/"mixed"/"green"}  ← кроме robot (нет texture_type)
        #   category_id: 1=cube, 2=sphere, 3=cylinder, 4=cone,
        #                5=torus, 6=capsule, 7=ellipsoid, 8=pyramid
        #                9=robot (фон, пропускается при загрузке)
        #   Реальный класс = NBV_PRIMITIVE_NAMES[category_id] + '_' + texture_type
        #   Итого: 8 форм × 3 текстуры = 24 класса (0-23 в my_train_odin.py)
        # ----------------------------------------------------------------
        # rgb/: 00000.png ... 00004.png  (5 кадров, 224×224)
        # depth/: 00000.npy ... 00004.npy
        # masks/: 00000.png ... 00004.png
        # labels/: 00000.txt ... 00004.txt
        # ================================================================

        # Проверим cameras.json
        cameras_path = sample_dir / "cameras.json"
        if cameras_path.exists():
            with open(cameras_path) as f:
                cameras = json.load(f)
            print(f"\n=== cameras.json ===")
            print(f"Тип: {type(cameras).__name__} (dict, ключ = индекс кадра)")
            print(f"Ключи кадров: {list(cameras.keys())}")
            first_key = list(cameras.keys())[0]
            first_cam = cameras[first_key]
            print(f"Поля кадра '{first_key}': {list(first_cam.keys())}")
            intr = first_cam["intrinsics"]
            print(f"Интринзика: fx={intr['fx']:.2f}, fy={intr['fy']:.2f}, cx={intr['cx']}, cy={intr['cy']}")
            print(f"Размер изображения: {intr['width']}x{intr['height']}")
            print(f"rotation (quat [x,y,z,w]): {first_cam['rotation']}")
            print(f"[OK] cx*2={intr['cx']*2}, cy*2={intr['cy']*2} → img размер из intrinsics корректен")

        # Проверим color_map.json
        color_map_path = sample_dir / "color_map.json"
        if color_map_path.exists():
            with open(color_map_path) as f:
                color_map = json.load(f)
            print(f"\n=== color_map.json ===")
            print(f"Тип данных: {type(color_map).__name__} (список объектов)")
            print(f"Количество объектов в сцене: {len(color_map)}")

            # Маппинг из my_train_odin.py
            NBV_PRIMITIVE_NAMES_DIAG = {
                1: "cube", 2: "sphere", 3: "cylinder", 4: "cone",
                5: "torus", 6: "capsule", 7: "ellipsoid", 8: "pyramid"
            }

            print("\nОбъекты в сцене:")
            unique_classes = set()
            for entry in color_map:
                cat_id = entry.get("category_id")
                texture = entry.get("texture_type", "N/A")
                inst_id = entry.get("instance_id")
                color = entry.get("color")

                if cat_id == 9:
                    cls_label = "robot (category_id=9, background → пропускается)"
                elif cat_id in NBV_PRIMITIVE_NAMES_DIAG:
                    cls_label = f"{NBV_PRIMITIVE_NAMES_DIAG[cat_id]}_{texture}"
                    unique_classes.add(cls_label)
                else:
                    cls_label = f"unknown_cat_{cat_id}"

                print(f"  inst={inst_id}, color={color}, cat_id={cat_id}, "
                      f"texture='{texture}' → class='{cls_label}'")

            print(f"\n=== Уникальные классы объектов в сцене: {len(unique_classes)} из 24 ===")
            for cls in sorted(unique_classes):
                print(f"  - {cls}")

            print("\n[ВАЖНО] 'category_name' всегда = 'target_object' — НЕ информативно!")
            print("[ВАЖНО] Реальный класс = NBV_PRIMITIVE_NAMES[category_id] + '_' + texture_type")
            print("[OK]    get_nbv_stage2_dataset_dicts() в my_train_odin.py правильно это обрабатывает")

        # Проверим количество кадров
        rgb_dir = sample_dir / "rgb"
        if rgb_dir.exists():
            rgb_files = sorted(rgb_dir.iterdir())
            print(f"\n=== Количество кадров: {len(rgb_files)} ===")
            print(f"  Файлы: {[f.name for f in rgb_files]}")
            print(f"[ВАЖНО] --num_frames должен быть <= {len(rgb_files)} (кадров на сэмпл)")

else:
    print("ERROR: DATASET_DIR не найден!")
    print("Проверьте, что датасет подключен в Kaggle Notebook")
    print("\nПопытка найти датасет автоматически...")

    if os.path.exists("/kaggle/input"):
        print("\nДоступные датасеты в /kaggle/input:")
        for root, dirs, files in os.walk("/kaggle/input"):
            level = root.replace("/kaggle/input", "").count(os.sep)
            if level < 3:
                indent = "  " * level
                print(f"{indent}{os.path.basename(root)}/")


## 3. Запуск обучения с SWAG

### Особенности датасета NBV Stage2 (vs Strawberry):
- **24 класса**: 8 примитивов × 3 текстуры (cube_red, cube_mixed, cube_green, sphere_red, ...)
- **Нет препятствий**: Stage 2 проще чем Stage 3
- **5 кадров** на сэмпл (не ~20 views как у Strawberry!)
- **Размер изображений**: 224×224 (меньше чем 640×480 у Strawberry)
- **cameras.json**: dict-формат (ключ = индекс кадра), НЕТ поля `frame_index` внутри
- **color_map.json**: list-формат, тип = `NBV_PRIMITIVE_NAMES[category_id] + '_' + texture_type`

> **Важно**: `--num_frames 5` — это максимум (все кадры). Можно уменьшить для экономии VRAM, но не стоит увеличивать выше 5.

In [ ]:
import shutil

def sync_previous_output():
    """Копирует старые чекпоинты для --resume"""
    input_base = "/kaggle/input"
    target_output = "./output_nbv_stage2"
    
    if not os.path.exists(input_base):
        return

    for root, dirs, files in os.walk(input_base):
        if "output" in root and any(f.endswith(".pth") for f in files):
            print(f"Нашел старые чекпоинты в: {root}")
            if not os.path.exists(target_output):
                os.makedirs(target_output)
            
            for f in files:
                src = os.path.join(root, f)
                dst = os.path.join(target_output, f)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
            print(f"Успешно скопировал {len(files)} файлов в {target_output}")
            return

sync_previous_output()

# Используем тот же DATASET_DIR что и в предыдущей ячейке
DATASET_DIR = "/kaggle/input/datasets/sergeykurchev/nbv-stage2-dataset"
SPLITS_FILE = "./nbv_stage2_splits.json"
CONFIG_FILE = "my_odin/configs/scannet_context/3d.yaml"

train_cmd = [
    VENV_PYTHON, "my_odin/my_train_odin.py",
    "--config-file", CONFIG_FILE,
    "--num-gpus", "1",
    "--dist-url", "tcp://127.0.0.1:23456",
    "--resume",
    "--visualize",
    "--dataset_dir", DATASET_DIR,
    "--splits_file", SPLITS_FILE,
    
    # ПАРАМЕТРЫ ОБУЧЕНИЯ
    "--num_epochs", "15",  # Больше эпох для NBV
    "--eval_period", "200",  # Валидация каждые 200 итераций
    "--checkpoint_period", "200",
    
    # ОГРАНИЧЕНИЕ ВРЕМЕНИ
    "--max_time", "6",
    
    # ПАРАМЕТРЫ РЕСУРСОВ
    "--image_size", "224",  # NBV использует 224×224
    "--batch_size", "2",  # Можно больше т.к. изображения меньше
    "--num_frames", "5",   # NBV Stage2 имеет ровно 5 кадров на сэмпл (00000..00004)
    "--lr", "0.0001",
    
    # ТЕХНИЧЕСКИЕ ПАРАМЕТРЫ
    'MODEL.WEIGHTS', CONFIG["ODIN_WEIGHTS_PATH"],
    'OUTPUT_DIR', './output_nbv_stage2',
    'SOLVER.AMP.ENABLED', 'True',
    
    # SWAG ПАРАМЕТРЫ
    'MODEL.BAYESIAN_TYPE', 'swag',
    'MODEL.BAYESIAN_SAMPLES', '1',
    'MODEL.BAYESIAN_INFERENCE_DURING_TRAINING', 'False',
    'MODEL.SWAG.START_EPOCH', '3',  # Начинаем раньше (15 эпох всего)
    'MODEL.SWAG.UPDATE_FREQ', '5',
    'MODEL.SWAG.MAX_MODELS', '10',
    'MODEL.SWAG.RANK', '20',
    'MODEL.SWAG.NO_COV_MAT', 'False',
]

print("Starting SWAG training on NBV Stage2 dataset...")
print("Dataset: Multi-object (2-10) with 3 textures")
print("Classes: Auto-detected (8 primitives + robot)")
print("Note: Dataset type and NUM_CLASSES are auto-detected from data format")

venv_env = make_venv_env()
run_in_venv(train_cmd, env=venv_env)

## 4. Inference с SWAG (после обучения)

Запустите эту ячейку после завершения обучения для полного Bayesian inference с uncertainty estimation.

In [ ]:
inference_cmd = [
    VENV_PYTHON, "my_odin/my_train_odin.py",
    "--config-file", CONFIG_FILE,
    "--num-gpus", "1",
    "--eval-only",  # Только inference
    "--dataset_dir", DATASET_DIR,
    "--splits_file", SPLITS_FILE,
    
    'MODEL.WEIGHTS', './output_nbv_stage2/model_final.pth',
    'OUTPUT_DIR', './output_nbv_stage2',
    
    # SWAG INFERENCE
    'MODEL.BAYESIAN_TYPE', 'swag',
    'MODEL.BAYESIAN_SAMPLES', '10',  # 10 сэмплов из posterior
    'MODEL.SWAG.SCALE', '1.0',
]

print("Running SWAG inference with uncertainty estimation...")
print("Note: NUM_CLASSES auto-detected from dataset format")
venv_env = make_venv_env()
run_in_venv(inference_cmd, env=venv_env)

## Результаты

Метрики сохраняются в:
- `output_nbv_stage2/metrics_comparison.csv`
- `output_nbv_stage2/swag_state.pth` - SWAG статистика
- `output_nbv_stage2/model_final.pth` - финальная модель

### Ожидаемые результаты:
- **Stage 2 проще** чем Stage 3 (нет окклюзий)
- **24 класса**: 8 примитивов × 3 текстуры
- **mAP**: ~70-80% (выше чем у Stage 3)
- **Uncertainty**: ниже чем в Stage 3 (меньше окклюзий)